# Project: Audience Segmentation for E-commerce

#### RFM Analysis

**Goal:** turn the cleaned, purchase-level data from `01_data_cleaning_eda.ipynb` into a **Recency–Frequency–Monetary (RFM)** table, score each customer 1–5 on each dimension, and group customers into actionable segments (Champions, At Risk, Hibernating, etc.).

**Workflow:**
1. Setup
2. Load clean data
3. Build the RFM table
4. Score customers (quantile-based RFM)
5. Coarse segmentation (Low / Mid / High value)
6. Granular segmentation (customer personas)
7. Segment overview & visualizations
8. Deep dive: VIP customers
9. Key insights & export

## 1. Setup

In [27]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Consistent pastel palette used across every categorical chart in this notebook
PASTEL = px.colors.qualitative.Pastel

## 2. Load Clean Data

In [28]:
# Load the cleaned data produced by the cleaning notebook.
# It includes BOTH purchases and cancellations (flagged via IsCancellation) -
# excludes only duplicates and rows with no CustomerID - and carries the ReferenceDate
# computed there so Recency stays anchored to that exact cleaned dataset.
data = pd.read_csv("data/online_retail_clean.csv", parse_dates=["InvoiceDate", "ReferenceDate"])
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,OrderDate,IsCancellation,ReferenceDate
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12-01,False,2011-12-10 12:50:00
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,False,2011-12-10 12:50:00
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12-01,False,2011-12-10 12:50:00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,False,2011-12-10 12:50:00
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12-01,False,2011-12-10 12:50:00


In [29]:
data.shape

(401604, 12)

In [30]:
# All rows share the same ReferenceDate
reference_date = data["ReferenceDate"].iloc[0]
print(f"Reference date: {reference_date}")

Reference date: 2011-12-10 12:50:00


## 3. Build the RFM Table

Aggregate from one row per line item to **one row per customer**:
- **Recency** — days since the customer's most recent *purchase* (cancellations excluded — a return isn't a new order and shouldn't make a customer look more active than they are)
- **Frequency** — number of distinct purchase orders (cancellations excluded, same reasoning)
- **TotalQuantity** — total units purchased (purchases only, same reasoning as Frequency)
- **Country** — the customer's country (most frequent value across their purchase rows, in case of any inconsistency)
- **GrossMonetary** — total spent on purchases only, ignoring any later cancellations
- **Monetary** — **net** amount spent: `GrossMonetary` minus matching cancellations. This is the figure used for RFM scoring and segmentation below.
- **CancelledAmount** — `GrossMonetary - Monetary`, i.e. how much of that gross spend came back as a return
- **ReturnRate** — `CancelledAmount / GrossMonetary`, i.e. what share of a customer's gross spend they later cancelled

In [31]:
purchases = data[~data["IsCancellation"]].copy()

rfm = purchases.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),          # distinct orders, not line-item count
    TotalQuantity=("Quantity", "sum"),            # purchases only
    Country=("Country", lambda x: x.mode().iloc[0]),  # most frequent country for this customer
    GrossMonetary=("TotalAmount", "sum")          # purchases only
).reset_index()

# Monetary sums TotalAmount across ALL rows (purchases + cancellations) per customer.
# Cancellation rows carry negative TotalAmount, so this automatically nets a customer's
# purchases against whatever they later returned - this IS the net figure, so it's stored
# directly as "Monetary" rather than as a separate "NetMonetary" column.
monetary_net = data.groupby("CustomerID")["TotalAmount"].sum().rename("Monetary")
rfm = rfm.merge(monetary_net, on="CustomerID", how="left")

rfm["CancelledAmount"] = rfm["GrossMonetary"] - rfm["Monetary"]
rfm["ReturnRate"] = (rfm["CancelledAmount"] / rfm["GrossMonetary"]).round(4)

# A handful of customers can return more, within this window, than they purchased in it
# (e.g. an item bought just before the data's start date) - Monetary can't sensibly go
# negative for scoring purposes, so we floor it at 0 and report how many were affected.
negative_net = (rfm["Monetary"] < 0).sum()
print(f"Customers with net Monetary < 0 (floored to 0 for scoring): {negative_net}")

rfm["Monetary"] = rfm["Monetary"].clip(lower=0)

rfm.head()

Customers with net Monetary < 0 (floored to 0 for scoring): 9


,CustomerID,Recency,Frequency,TotalQuantity,Country,GrossMonetary,Monetary,CancelledAmount,ReturnRate
0,12346,326,1,74215,United Kingdom,77183.60,0.00,77183.6,1.0
1,12347,2,7,2458,Iceland,4310.00,4310.00,0.0,0.0
2,12348,75,4,2341,Finland,1797.24,1797.24,0.0,0.0
3,12349,19,1,631,Italy,1757.55,1757.55,0.0,0.0
4,12350,310,1,197,Norway,334.40,334.40,0.0,0.0


In [32]:
# Sanity check: does netting actually change anything, or is it mostly a formality?
fig = px.scatter(
    rfm,
    x="GrossMonetary",
    y="Monetary",
    hover_data=["CustomerID", "ReturnRate"],
    title="Gross vs. Net Monetary per Customer",
    labels={"GrossMonetary": "Gross Monetary (purchases only)", "Monetary": "Monetary (net of cancellations)"}
)
fig.add_shape(type="line", x0=0, y0=0, x1=rfm["GrossMonetary"].max(), y1=rfm["GrossMonetary"].max(),
              line=dict(color="lightgrey", dash="dash"))
fig.update_traces(marker_color=PASTEL[0])
fig.update_layout(template="simple_white")
fig.show()

Points on the dashed line returned nothing; points below it had at least one cancellation pulling Net below Gross. `CustomerID` 12346 — flagged earlier as a single ~£77K order that was fully cancelled — should now sit at (or very near) Monetary = 0, exactly where we'd want it.

In [33]:
rfm[["Recency", "Frequency", "GrossMonetary", "Monetary", "ReturnRate"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Recency,4339.0,92.518322,100.009747,1.0,18.000,51.00,142.0000,374.0000
Frequency,4339.0,4.271952,7.705493,1.0,1.000,2.00,5.0000,210.0000
GrossMonetary,4339.0,2048.215924,8984.248352,0.0,306.455,668.56,1660.3150,280206.0200
Monetary,4339.0,1910.981462,8247.155109,0.0,297.975,649.50,1619.6800,279489.0200
ReturnRate,4338.0,0.039677,0.874860,0.0,0.000,0.00,0.0108,57.0507


**Observation** — Frequency and Monetary are both right-skewed (their max values sit far above the 75th percentile), while Recency is skewed too, just less severely (mean well above median). This is exactly why quantile-based scoring is used instead of fixed-value thresholds: it splits customers into even 20% bands based on relative rank, so a handful of extreme bulk buyers can't drag the score boundaries and leave most customers stuck in one bucket. 

## 4. Score Customers (Quantile-Based RFM Scores)

Each customer gets a score of 1 (worst) to 5 (best) on each dimension, based on which quintile (20% band) of the customer base they fall into. This gives a balanced distribution of customers across score ranges.

In [34]:
quantiles = rfm[["Recency", "Frequency", "Monetary"]].quantile(q=[0.2, 0.4, 0.6, 0.8])


def quantile_score(value, thresholds, reverse=False):
    """
    Map a raw value to a 1-5 score based on its quintile bucket.

    reverse=False (Frequency, Monetary): higher raw value -> higher score (5 = best).
    reverse=True  (Recency): lower raw value -> higher score, since a SMALLER
    number of days since last purchase is better for the customer.
    """
    score = 1
    for q in (0.2, 0.4, 0.6, 0.8):
        if value <= thresholds[q]:
            break
        score += 1
    return (6 - score) if reverse else score


rfm["R"] = rfm["Recency"].apply(quantile_score, thresholds=quantiles["Recency"], reverse=True)
rfm["F"] = rfm["Frequency"].apply(quantile_score, thresholds=quantiles["Frequency"])
rfm["M"] = rfm["Monetary"].apply(quantile_score, thresholds=quantiles["Monetary"])

rfm.head()

,CustomerID,Recency,Frequency,TotalQuantity,Country,GrossMonetary,Monetary,CancelledAmount,ReturnRate,R,F,M
0,12346,326,1,74215,United Kingdom,77183.60,0.00,77183.6,1.0,1,1,1
1,12347,2,7,2458,Iceland,4310.00,4310.00,0.0,0.0,5,5,5
2,12348,75,4,2341,Finland,1797.24,1797.24,0.0,0.0,2,4,4
3,12349,19,1,631,Italy,1757.55,1757.55,0.0,0.0,4,1,4
4,12350,310,1,197,Norway,334.40,334.40,0.0,0.0,1,1,2


In [35]:
# RFM_Segment keeps the individual scores as a 3-digit code (e.g. "543") for quick lookup,
# RFM_Score sums them into a single 3-15 scale for coarse ranking
rfm["RFM_Segment"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)
rfm["RFM_Score"] = rfm[["R", "F", "M"]].sum(axis=1)

rfm.head()

,CustomerID,Recency,Frequency,TotalQuantity,Country,GrossMonetary,Monetary,CancelledAmount,ReturnRate,R,F,M,RFM_Segment,RFM_Score
0,12346,326,1,74215,United Kingdom,77183.60,0.00,77183.6,1.0,1,1,1,111,3
1,12347,2,7,2458,Iceland,4310.00,4310.00,0.0,0.0,5,5,5,555,15
2,12348,75,4,2341,Finland,1797.24,1797.24,0.0,0.0,2,4,4,244,10
3,12349,19,1,631,Italy,1757.55,1757.55,0.0,0.0,4,1,4,414,9
4,12350,310,1,197,Norway,334.40,334.40,0.0,0.0,1,1,2,112,4


In [36]:
# Sanity-check the top-scoring band: "555" score just means recent + frequent + high-spend.
top_scorers = rfm.loc[
    rfm["RFM_Segment"] == "555",
    ["CustomerID", "Recency", "Frequency", "Monetary", "ReturnRate", "R", "F", "M", "RFM_Segment"]
].sort_values("Monetary", ascending=False)

print(f"{len(top_scorers):,} customers scored 555 (top band on all three dimensions)")
top_scorers.head()

322 customers scored 555 (top band on all three dimensions)


,CustomerID,Recency,Frequency,Monetary,ReturnRate,R,F,M,RFM_Segment
1690,14646,2,74,279489.02,0.0026,5,5,5,555
4202,18102,1,60,256438.49,0.0124,5,5,5,555
3729,17450,8,46,187322.17,0.0364,5,5,5,555
1880,14911,1,201,132458.73,0.0783,5,5,5,555
1334,14156,10,55,113214.59,0.0341,5,5,5,555


## 5. Coarse Segmentation (Low / Mid / High Value)

A quick, single-number view before the more detailed persona segmentation below. `RFM_Score` ranges 3-15 (three dimensions x 1-5 each); we split it into three roughly even bands.

In [37]:
def assign_value_segment(score):
    if score < 5:
        return "Low-value"
    elif score < 10:
        return "Mid-value"
    else:
        return "High-value"


rfm["ValueSegment"] = rfm["RFM_Score"].apply(assign_value_segment)
rfm.head()

,CustomerID,Recency,Frequency,TotalQuantity,Country,GrossMonetary,Monetary,CancelledAmount,ReturnRate,R,F,M,RFM_Segment,RFM_Score,ValueSegment
0,12346,326,1,74215,United Kingdom,77183.60,0.00,77183.6,1.0,1,1,1,111,3,Low-value
1,12347,2,7,2458,Iceland,4310.00,4310.00,0.0,0.0,5,5,5,555,15,High-value
2,12348,75,4,2341,Finland,1797.24,1797.24,0.0,0.0,2,4,4,244,10,High-value
3,12349,19,1,631,Italy,1757.55,1757.55,0.0,0.0,4,1,4,414,9,Mid-value
4,12350,310,1,197,Norway,334.40,334.40,0.0,0.0,1,1,2,112,4,Low-value


In [ ]:
# Llogical order (not alphabetical) for chart display
VALUE_SEGMENT_ORDER = ["Low-value", "Mid-value", "High-value"]

# Numeric rank so Power BI can sort ValueSegment correctly, not alphabetically
rfm["ValueSegmentRank"] = rfm["ValueSegment"].map({seg: i + 1 for i, seg in enumerate(VALUE_SEGMENT_ORDER)})

value_segment_counts = (
    rfm["ValueSegment"]
    .value_counts()
    .reindex(VALUE_SEGMENT_ORDER)
    .reset_index()
)
value_segment_counts.columns = ["ValueSegment", "Count"]
value_segment_counts

,ValueSegment,Count
0,Low-value,707
1,Mid-value,1862
2,High-value,1770


In [39]:
fig = px.bar(
    value_segment_counts,
    x="ValueSegment",
    y="Count",
    title="Customer Distribution by Value Segment",
    labels={"ValueSegment": "Value Segment", "Count": "Number of Customers"},
    color="ValueSegment",
    category_orders={"ValueSegment": VALUE_SEGMENT_ORDER},
    color_discrete_sequence=PASTEL
)
fig.update_layout(template="simple_white", showlegend=False)
fig.show()

## 6. Granular Segmentation (Customer Personas)

#### Segment Definitions

| Segment | Description | Logic |
| --- | --- | --- |
| **Champions** | Best customers: very recent, frequent, and high-value buyers. | `R ≥ 5` and `F ≥ 4` and `M ≥ 4` |
| **Loyal Customers** | High-value, frequent customers who remain relatively active. | `R ≥ 3` and `F ≥ 4` and `M ≥ 4` |
| **Do Not Lose** | Highly valuable and frequent customers who have become inactive. | `R ≤ 2` and `F ≥ 4` and `M ≥ 4` |
| **At Risk** | Previously valuable and engaged customers who are now inactive. | `R ≤ 2` and `F ≥ 3` and `M ≥ 3` |
| **Need Attention** | Moderate recency with strong engagement/value; may need attention. | `R = 3` and `F ≥ 3` and `M ≥ 3` |
| **Potential Champions** | Recent customers with moderate-to-high frequency and value. | `R ≥ 4` and `F ≥ 2` and `M ≥ 2` |
| **About to Sleep** | Moderate recency but low engagement and value; may soon go inactive. | `R = 3` and `F ≤ 2` and `M ≤ 2` |
| **Promising** | Recent customers with low frequency and value, but showing potential. | `R = 4` and `F ≤ 2` and `M ≤ 2` |
| **New Customers** | Very recent, newly acquired customers with low frequency and value. | `R = 5` and `F ≤ 2` and `M ≤ 2` |
| **Hibernating** | Old customers with low frequency or monetary value. | everything else |

Rules are checked in this order, so more specific/valuable segments (e.g. Champions) are matched before broader ones (e.g. Potential Champions) — a customer is assigned to the first rule they satisfy.

In [40]:
def assign_customer_persona(row):
    R, F, M = row["R"], row["F"], row["M"]

    if R >= 5 and F >= 4 and M >= 4:
        return "Champions"
    elif R >= 3 and F >= 4 and M >= 4:
        return "Loyal Customers"
    elif R <= 2 and F >= 4 and M >= 4:
        return "Do Not Lose"
    elif R <= 2 and F >= 3 and M >= 3:
        return "At Risk"
    elif R == 3 and F >= 3 and M >= 3:
        return "Need Attention"
    elif R >= 4 and F >= 2 and M >= 2:
        return "Potential Champions"
    elif R == 3 and F <= 2 and M <= 2:
        return "About to Sleep"
    elif R == 4 and F <= 2 and M <= 2:
        return "Promising"
    elif R == 5 and F <= 2 and M <= 2:
        return "New Customers"
    else:
        return "Hibernating"


rfm["CustomerPersona"] = rfm.apply(assign_customer_persona, axis=1)
rfm.head(10)

,CustomerID,Recency,Frequency,TotalQuantity,Country,GrossMonetary,Monetary,CancelledAmount,ReturnRate,R,F,M,RFM_Segment,RFM_Score,ValueSegment,ValueSegmentRank,CustomerPersona
0,12346,326,1,74215,United Kingdom,77183.60,0.00,77183.60,1.0000,1,1,1,111,3,Low-value,1,Hibernating
1,12347,2,7,2458,Iceland,4310.00,4310.00,0.00,0.0000,5,5,5,555,15,High-value,3,Champions
2,12348,75,4,2341,Finland,1797.24,1797.24,0.00,0.0000,2,4,4,244,10,High-value,3,Do Not Lose
3,12349,19,1,631,Italy,1757.55,1757.55,0.00,0.0000,4,1,4,414,9,Mid-value,2,Hibernating
4,12350,310,1,197,Norway,334.40,334.40,0.00,0.0000,1,1,2,112,4,Low-value,1,Hibernating
5,12352,36,8,536,Norway,2506.04,1545.41,960.63,0.3833,3,5,4,354,12,High-value,3,Loyal Customers
6,12353,204,1,20,Bahrain,89.00,89.00,0.00,0.0000,1,1,1,111,3,Low-value,1,Hibernating
7,12354,232,1,530,Spain,1079.40,1079.40,0.00,0.0000,1,1,4,114,6,Mid-value,2,Hibernating
8,12355,214,1,240,Bahrain,459.40,459.40,0.00,0.0000,1,1,2,112,4,Low-value,1,Hibernating
9,12356,23,3,1591,Portugal,2811.43,2811.43,0.00,0.0000,4,3,5,435,12,High-value,3,Potential Champions


In [41]:
# Business-priority order (best -> worst) - defined once here, reused for chart ordering,
# the VIP filter below, and as a numeric rank Power BI can sort CustomerPersona by
# (alphabetically, "About to Sleep" would sort before "Champions", which isn't useful)
PERSONA_ORDER = [
    "Champions", "Loyal Customers", "Potential Champions", "New Customers", "Promising",
    "Need Attention", "About to Sleep", "At Risk", "Do Not Lose", "Hibernating"
]
VIP_PERSONAS = ["Champions", "Loyal Customers"]

rfm["PersonaRank"] = rfm["CustomerPersona"].map({seg: i + 1 for i, seg in enumerate(PERSONA_ORDER)})
rfm.head()

,CustomerID,Recency,Frequency,TotalQuantity,Country,GrossMonetary,Monetary,CancelledAmount,ReturnRate,R,F,M,RFM_Segment,RFM_Score,ValueSegment,ValueSegmentRank,CustomerPersona,PersonaRank
0,12346,326,1,74215,United Kingdom,77183.60,0.00,77183.6,1.0,1,1,1,111,3,Low-value,1,Hibernating,10
1,12347,2,7,2458,Iceland,4310.00,4310.00,0.0,0.0,5,5,5,555,15,High-value,3,Champions,1
2,12348,75,4,2341,Finland,1797.24,1797.24,0.0,0.0,2,4,4,244,10,High-value,3,Do Not Lose,9
3,12349,19,1,631,Italy,1757.55,1757.55,0.0,0.0,4,1,4,414,9,Mid-value,2,Hibernating,10
4,12350,310,1,197,Norway,334.40,334.40,0.0,0.0,1,1,2,112,4,Low-value,1,Hibernating,10


## 7. Segment Overview & Visualizations

In [ ]:
# Count the number of customers per CustomerPersona
persona_counts = rfm["CustomerPersona"].value_counts().reset_index()
persona_counts.columns = ["CustomerPersona", "Count"]
persona_counts = persona_counts.sort_values("Count", ascending=False)
persona_counts

,CustomerPersona,Count
0,Hibernating,1628
1,Loyal Customers,591
2,Potential Champions,566
3,Champions,548
4,About to Sleep,307
5,At Risk,181
6,Promising,170
7,Do Not Lose,139
8,Need Attention,135
9,New Customers,74


In [50]:
persona_value = (
    rfm.groupby("CustomerPersona")
    .agg(
        GrossMonetary=("GrossMonetary", "sum"),
        Monetary=("Monetary", "sum"),
        TotalQuantity=("TotalQuantity", "sum"),
        CancelledAmount=("CancelledAmount", "sum"),
        Customers=("CustomerID", "nunique"),
        Frequency=("Frequency", "mean"),
        Recency=("Recency", "mean")
    )
    .round(2).reset_index()
)

# Add total row
total = pd.DataFrame({
    "CustomerPersona": ["Total"],
    "GrossMonetary": [rfm["GrossMonetary"].sum()],
    "Monetary": [rfm["Monetary"].sum()],
    "TotalQuantity": [rfm["TotalQuantity"].sum()],
    "CancelledAmount": [rfm["CancelledAmount"].sum()],
    "Customers": [rfm["CustomerID"].nunique()],
    "Frequency": [rfm["Frequency"].mean()],
    "Recency": [rfm["Recency"].mean()]
}).round(2)

persona_value = pd.concat([persona_value, total], ignore_index=True)

persona_value

,CustomerPersona,GrossMonetary,Monetary,TotalQuantity,CancelledAmount,Customers,Frequency,Recency
0,About to Sleep,85382.03,78346.05,55397,7044.13,307,1.24,53.19
1,At Risk,288189.09,217978.83,142562,70210.26,181,3.44,139.95
2,Champions,4256107.27,4141438.86,2419924,114668.41,548,13.72,5.72
3,Do Not Lose,329932.92,324299.64,202381,5633.28,139,5.99,123.83
4,Hibernating,904746.48,807029.13,570624,100527.10,1628,1.44,183.77
5,Loyal Customers,2148312.37,2038357.66,1258141,109954.71,591,7.57,32.88
6,Need Attention,156247.25,145725.63,90653,10521.62,135,3.41,51.42
7,New Customers,184449.08,15685.37,93403,168763.71,74,1.23,7.18
8,Potential Champions,496837.79,486930.45,294976,9907.34,566,2.87,16.62
9,Promising,37004.61,35956.94,37825,1047.67,170,1.12,23.15


In [ ]:
fig_treemap = px.treemap(
    persona_counts,
    path=["CustomerPersona"],
    values="Count",
    color="CustomerPersona",
    color_discrete_sequence=PASTEL,
    title="Customer Personas by Customer Count"
)
fig_treemap.update_traces(textinfo="label+value+percent parent")
fig_treemap.update_layout(title_x=0.5)
fig_treemap.show()

In [ ]:
# Highlight VIP personas with a stronger pastel accent against the rest of the pastel palette
vip_accent = "#FFADAD"
bar_colors = [vip_accent if seg in VIP_PERSONAS else PASTEL[i % len(PASTEL)]
              for i, seg in enumerate(PERSONA_ORDER)]

ordered_counts = persona_counts.set_index("CustomerPersona").reindex(PERSONA_ORDER)["Count"]

fig = go.Figure(
    data=[go.Bar(
        x=PERSONA_ORDER,
        y=ordered_counts.values,
        marker_color=bar_colors,
        marker_line_color="rgb(140,140,140)",
        marker_line_width=1
    )]
)
fig.update_layout(
    title="Customer Distribution by Persona",
    xaxis_title="Customer Persona",
    yaxis_title="Number of Customers",
    template="simple_white",
    showlegend=False
)
fig.show()

In [ ]:
# Average R, F, M score per persona
segment_scores = (
    rfm.groupby("CustomerPersona")[["R", "F", "M"]]
    .mean()
    .reindex(PERSONA_ORDER)
    .reset_index()
)

fig = go.Figure()
for metric, color in zip(["R", "F", "M"], PASTEL[:3]):
    fig.add_trace(go.Bar(
        x=segment_scores["CustomerPersona"],
        y=segment_scores[metric],
        name={"R": "Recency Score", "F": "Frequency Score", "M": "Monetary Score"}[metric],
        marker_color=color
    ))

fig.update_layout(
    title="Average RFM Score by Persona",
    xaxis_title="Customer Persona",
    yaxis_title="Average Score (1-5)",
    barmode="group",
    template="simple_white"
)
fig.show()

## 8. Deep Dive: VIP Customers

Champions and Loyal Customers drive disproportionate revenue — worth a closer look at their spread and whether Recency, Frequency and Monetary move together within this group.

In [ ]:
vip_segment = rfm[rfm["CustomerPersona"].isin(VIP_PERSONAS)]
print(f"VIP customers: {len(vip_segment):,} ({len(vip_segment) / len(rfm):.1%} of all customers)")

VIP customers: 1,139 (26.3% of all customers)


In [ ]:
# Box plot to check the spread and outliers within the VIP segment
fig = go.Figure()
for metric, color in zip(["Recency", "Frequency", "Monetary"], PASTEL[:3]):
    fig.add_trace(go.Box(y=vip_segment[metric], name=metric, marker_color=color))

fig.update_layout(
    title="RFM Distribution of VIP Customers (Champions + Loyal)",
    yaxis_title="Value",
    template="simple_white"
)
fig.show()

**VIP customers** still show Monetary above £100K even after netting (e.g. the top scorer nets ~£280K across 74 orders). At this dataset's scale that reads as a wholesale/reseller account rather than fraud, so it isn't flagged as suspicious on its own — so worth knowing about.

In [ ]:
vip_segment[["Recency", "Frequency", "GrossMonetary", "Monetary", "ReturnRate"]].describe().T

,count,mean,std,min,25%,50%,75%,max
Recency,1139.0,19.813872,17.834243,1.00,5.000,15.0000,29.5000,72.0000
Frequency,1139.0,10.530290,12.902695,4.00,5.000,7.0000,11.0000,210.0000
GrossMonetary,1139.0,5622.844284,15980.463803,913.68,1663.475,2590.3100,4514.2500,280206.0200
Monetary,1139.0,5425.633468,15499.226294,913.68,1624.095,2535.8800,4406.4900,279489.0200
ReturnRate,1139.0,0.023892,0.057493,0.00,0.000,0.0069,0.0204,0.5714


In [ ]:
correlation_matrix = vip_segment[["R", "F", "M"]].corr()

fig_heatmap = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.columns,
    colorscale="RdBu",
    zmid=0,
    colorbar=dict(title="Correlation")
))
# Diverging colorscale kept here deliberately (not pastel) - correlation needs a
# scale that clearly distinguishes positive from negative, which pastel qualitative colors can't do.
fig_heatmap.update_layout(
    title="Correlation Matrix of R / F / M within VIP Customers",
    width=600,
    height=500
)
fig_heatmap.show()

**Insight**: within VIP customers, Recency shows little to no correlation with Frequency or Monetary — being a big spender or frequent buyer doesn't predict how recently a VIP last purchased. That means Recency needs to be monitored as an independent signal (a re-engagement trigger) rather than assumed to track with the other two.

## 9. Key Insights & Export

- **Dashboard-ready**: `TotalQuantity` and `Country` are included directly in `rfm` (not a separate merge) so the single exported file has everything Power BI needs — no join required on the BI side. `PersonaRank` and `ValueSegmentRank` are also included so Power BI can sort `CustomerPersona`/`ValueSegment` by business priority instead of alphabetically.
- **Reference date**: read directly from the cleaned file instead of recomputed, so this notebook always scores against the same cutoff the cleaning notebook used.
- **Frequency**: counted as *distinct purchase orders* (`InvoiceNo.nunique`, cancellations excluded), not line items — counting line items would have overstated how often multi-item shoppers buy.
- **Monetary is net, not gross**: `Monetary` = purchases minus matching cancellations. `GrossMonetary` is kept alongside it, plus `CancelledAmount` and `ReturnRate` so return behavior stays visible instead of disappearing into one number. 
- **Scoring approach**: quantile-based 1-5 scoring keeps the customer base evenly spread across score bands and already absorbs the right-skew in Frequency/Monetary, since it scores relative rank rather than raw magnitude.
- **Segments**: personas are assigned by evaluating rules in priority order, so the most valuable label a customer qualifies for wins (e.g. a customer meeting both "Champions" and "Potential Champions" criteria is labeled a Champion).
- **VIPs**: Champions + Loyal Customers make up a disproportionate share of value; within this group, Recency moves independently of Frequency/Monetary, so re-engagement timing should be tracked separately from spend/frequency health.

In [ ]:
# One row per customer, dashboard-ready: RFM metrics, scores, segments, plus TotalQuantity and Country
rfm.to_csv("data/customer_rfm_segments.csv", index=False)
print("Saved: data/customer_rfm_segments.csv")

Saved: data/customer_rfm_segments.csv
